# 2D joint histograms of registered 4D neutron / X-ray tomography

One 2D histogram per timepoint, saved as `.npy`, then every timepoint normalized
against the first to track how the neutron–X-ray relationship evolves.

**Three things that matter for the comparison to be meaningful:**

1. **Fixed bin edges.** Estimated once across *all* timepoints and reused. If each
   histogram got its own auto-range, bin *(i, j)* would mean something different at
   every `t` and comparing them would be nonsense.
2. **Count normalization before comparison.** Each histogram is divided by its own
   sum. Otherwise a change in how many voxels the mask admits looks exactly like a
   change in the distribution.
3. **Masking.** Air is usually the tallest peak in both modalities and will swamp
   everything the sample does.

Inputs are 4D TIFFs, read lazily (memory-mapped when the file allows it, otherwise
through a zarr view) so a full volume is never held in RAM. **Check the axis order
printed by section 2** — plain 4D TIFFs often carry no axis metadata and the
notebook has to fall back to assuming `TZYX`.

Requires `tifffile`; `zarr` is also needed for compressed or non-contiguous files:

```
pip install tifffile zarr
```

## 0. Setup

In [ ]:
import glob
import json
import os

import numpy as np
import matplotlib.pyplot as plt
import tifffile

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

## 1. Configuration

`VOL_A` / `VOL_B` may each be:

* a single 4D `.tif` / `.tiff` of shape `(T, Z, Y, X)`,
* a directory of 3D TIFFs, one per timepoint, sorted by filename, or
* the `.npy` equivalents of either.

`AXES_A` / `AXES_B` override the axis order when the file carries no usable
metadata. Leave them `None` to trust what `tifffile` reports.

In [ ]:
# --- inputs -----------------------------------------------------------------
VOL_A = "neutron_4d.tif"        # rows of the histogram
VOL_B = "xray_4d.tif"           # columns of the histogram
LABEL_A, LABEL_B = "neutron", "X-ray"

AXES_A = None                   # e.g. "TZYX" or "ZTYX" to override; None = auto
AXES_B = None

MASK = None                     # path to a 3D (static) or 4D (per-timepoint)
                                # mask, TIFF or .npy, nonzero = keep.
                                # Strongly recommended.

OUT_DIR = "hists"               # where the per-timepoint .npy files go

# --- binning ----------------------------------------------------------------
BINS = 256
RANGE_A = None                  # (lo, hi) to set explicitly, or None to estimate
RANGE_B = None
PLO, PHI = 0.05, 99.95          # percentiles used when a range is None
STRIDE = 4                      # subsampling factor for the range estimate
CLIP = True                     # clip out-of-range voxels into the edge bins
                                # (keeps total counts constant across timepoints)
CHUNK = 16                      # slices processed at a time

# --- comparison against t=0 -------------------------------------------------
MODE = "log2ratio"              # "diff" | "ratio" | "log2ratio" | "chi"
EPS = 1e-12
REF_FLOOR = 1e-6                # blank bins where t0 < this fraction of its max

os.makedirs(OUT_DIR, exist_ok=True)

## 2. Load the volumes

In [ ]:
TIFF_EXT = (".tif", ".tiff")


def open_lazy(path):
    """Open a TIFF or .npy lazily. Returns (array_like, axes, how)."""
    if path.lower().endswith(".npy"):
        arr = np.load(path, mmap_mode="r")
        return arr, None, "npy memmap"

    with tifffile.TiffFile(path) as tf:
        axes = tf.series[0].axes

    try:                                    # contiguous + uncompressed
        return tifffile.memmap(path), axes, "tiff memmap"
    except (ValueError, MemoryError):
        pass
    try:                                    # compressed / non-contiguous
        import zarr
        z = zarr.open(tifffile.imread(path, aszarr=True), mode="r")
        if not hasattr(z, "shape"):         # a group: take the first array
            z = z[list(z.array_keys())[0]]
        return z, axes, "tiff via zarr (lazy)"
    except ImportError:
        return (tifffile.imread(path), axes,
                "tiff FULLY IN RAM - pip install zarr for lazy access")


def resolve_axes(axes, shape, override, path, quiet=False):
    """Return an axis label string of len(shape), using '_' for singleton axes.

    TIFF axis metadata is only trusted when it explicitly names both T and Z.
    ImageJ hyperstacks routinely label the time axis Z and the depth axis C,
    and plain 4D TIFFs carry no labels at all, so anything else is resolved
    from the shape: the last two axes are Y and X, and the non-singleton
    leading axes are taken as T then Z in stored order.
    """
    n = len(shape)
    if override:
        if len(override) != n:
            raise ValueError(f"{path}: override {override!r} has {len(override)} "
                             f"labels but array has {n} dims")
        return override.upper()

    if (axes and len(axes) == n and "T" in axes and "Z" in axes
            and all(a in "TZYX" or shape[i] == 1 for i, a in enumerate(axes))):
        return axes

    if n < 3:
        raise ValueError(f"{path}: need at least 3 dims, got shape {shape}")

    big = [i for i in range(n - 2) if shape[i] > 1]
    labels = ["_"] * n
    labels[-2], labels[-1] = "Y", "X"
    if len(big) == 2:
        labels[big[0]], labels[big[1]] = "T", "Z"
    elif len(big) == 1:
        labels[big[0]] = "Z"          # a single timepoint
    else:
        raise ValueError(f"{path}: cannot resolve {len(big)} leading axes of "
                         f"shape {shape} - set an explicit override")
    out = "".join(labels)
    if not quiet:
        print(f"  ! {os.path.basename(path)}: reported axes {axes!r} not usable - "
              f"resolved as {out} from shape {tuple(shape)}")
    return out


class _Vol:
    """Lazy 3D view of one timepoint. Slices reach the file, not RAM."""

    def __init__(self, parent, t):
        self.p, self.t = parent, t
        self.shape = parent.vol_shape

    def __getitem__(self, key):
        if not isinstance(key, tuple):
            key = (key,)
        key = key + (slice(None),) * (3 - len(key))
        idx = [slice(None)] * len(self.p.axes)
        if self.p.t_axis is not None:
            idx[self.p.t_axis] = self.t
        for ax, v in self.p.fixed.items():
            idx[ax] = v
        for ax, k in zip(self.p.spatial, key):
            idx[ax] = k
        return np.asarray(self.p.data[tuple(idx)])


class Series:
    """N-D array + axis labels, presented as a sequence of 3D volumes."""

    def __init__(self, data, axes):
        self.data, self.axes = data, axes
        self.t_axis = axes.index("T") if "T" in axes else None
        self.spatial = [i for i, a in enumerate(axes) if a in "ZYX"]
        if len(self.spatial) != 3:
            raise ValueError(f"expected Z, Y and X axes, got {axes!r}")
        self.fixed = {}
        for i, a in enumerate(axes):
            if a in "TZYX":
                continue
            if data.shape[i] != 1:
                raise ValueError(f"axis {a!r} has size {data.shape[i]} > 1 "
                                 f"(unexpected non-spatial axis) - set an explicit override")
            self.fixed[i] = 0

    def __len__(self):
        return 1 if self.t_axis is None else self.data.shape[self.t_axis]

    def __getitem__(self, t):
        return _Vol(self, t)

    @property
    def vol_shape(self):
        return tuple(self.data.shape[i] for i in self.spatial)


class FileSeries:
    """One 3D file per timepoint."""

    def __init__(self, files, override):
        self.vols = []
        for n, f in enumerate(files):
            data, axes, _ = open_lazy(f)
            self.vols.append(
                Series(data, resolve_axes(axes, data.shape, override, f, quiet=n > 0)))
        shapes = {v.vol_shape for v in self.vols}
        if len(shapes) != 1:
            raise ValueError(f"timepoint volumes differ in shape: {shapes}")

    def __len__(self):
        return len(self.vols)

    def __getitem__(self, t):
        return self.vols[t][0]

    @property
    def vol_shape(self):
        return self.vols[0].vol_shape


def load_series(path, override=None):
    if os.path.isdir(path):
        files = sorted(sum((glob.glob(os.path.join(path, f"*{e}"))
                            for e in TIFF_EXT + (".npy",)), []))
        if not files:
            raise FileNotFoundError(f"No TIFF or .npy files found in {path}")
        print(f"  {path}: {len(files)} files, one per timepoint")
        return FileSeries(files, override)

    data, axes, how = open_lazy(path)
    print(f"  {path}: shape {tuple(data.shape)}, {how}")
    return Series(data, resolve_axes(axes, data.shape, override, path))


def load_mask(path, n_t, vol_shape):
    """Returns mask_of(t) -> 3D bool array (or None when there is no mask)."""
    if path is None:
        return lambda t: None
    data, axes, _ = open_lazy(path)
    axes = resolve_axes(axes, data.shape, None, path)
    s = Series(data, axes)
    if tuple(s.vol_shape) != tuple(vol_shape):
        raise ValueError(f"mask volume shape {s.vol_shape} != {vol_shape}")
    if s.t_axis is None:
        static = np.asarray(s[0][:]).astype(bool)
        return lambda t: static
    if len(s) != n_t:
        raise ValueError(f"mask has {len(s)} timepoints, volumes have {n_t}")
    return lambda t: s[t]


series_a = load_series(VOL_A, AXES_A)
series_b = load_series(VOL_B, AXES_B)

assert len(series_a) == len(series_b), f"timepoint mismatch: {len(series_a)} vs {len(series_b)}"
assert tuple(series_a.vol_shape) == tuple(series_b.vol_shape), (
    f"volumes are not on the same grid: {series_a.vol_shape} vs {series_b.vol_shape}"
)

N_T = len(series_a)
mask_of = load_mask(MASK, N_T, series_a.vol_shape)

print(f"\naxes A: {series_a.axes}   axes B: {series_b.axes}")
print(f"{N_T} timepoints, volume shape (Z, Y, X) = {tuple(series_a.vol_shape)}")
print(f"dtype A: {series_a[0][:1].dtype}   dtype B: {series_b[0][:1].dtype}")
print(f"mask: {MASK or 'none (air will dominate the histogram)'}")

**Confirm the axis assignment above before continuing.** If the number of timepoints
and the first spatial dimension look swapped, the file's `T` and `Z` are the other
way round — set `AXES_A = "ZTYX"` (or whatever matches) and re-run.

## 3. Bin edges

Estimated from a strided subsample of every timepoint, not just the first —
attenuation values drift as the sample evolves, and the bins have to cover the
whole temporal dynamic range.

In [ ]:
def estimate_range(series, plo, phi, stride, mask_of=None):
    chunks = []
    for t in range(len(series)):
        v = np.asarray(series[t][::stride, ::stride, ::stride], dtype=np.float64)
        if mask_of is not None:
            m = mask_of(t)
            if m is not None:
                v = v[np.asarray(m[::stride, ::stride, ::stride]).astype(bool)]
        v = v[np.isfinite(v)]
        chunks.append(v.ravel())
    s = np.concatenate(chunks)
    lo, hi = np.percentile(s, [plo, phi])
    if hi <= lo:
        lo, hi = float(s.min()), float(s.max())
    return float(lo), float(hi)


lo_a, hi_a = RANGE_A if RANGE_A else estimate_range(series_a, PLO, PHI, STRIDE, mask_of)
lo_b, hi_b = RANGE_B if RANGE_B else estimate_range(series_b, PLO, PHI, STRIDE, mask_of)

edges_a = np.linspace(lo_a, hi_a, BINS + 1)
edges_b = np.linspace(lo_b, hi_b, BINS + 1)
extent = [edges_b[0], edges_b[-1], edges_a[0], edges_a[-1]]

np.save(os.path.join(OUT_DIR, "edges_a.npy"), edges_a)
np.save(os.path.join(OUT_DIR, "edges_b.npy"), edges_b)

print(f"{LABEL_A} (rows): [{lo_a:.6g}, {hi_a:.6g}]  {BINS} bins")
print(f"{LABEL_B} (cols): [{lo_b:.6g}, {hi_b:.6g}]  {BINS} bins")

## 4. Sanity check on the first timepoint

Compute just `t=0` and look at it before running the whole series. What you want to
see: the joint distribution filling a decent fraction of the frame. Everything
crushed into one corner means the ranges are too wide; hard bright lines along the
first/last row or column mean clipping is piling up out-of-range voxels there, so
set `RANGE_A`/`RANGE_B` explicitly or widen the percentiles.

In [ ]:
def joint_hist(vol_a, vol_b, edges_a, edges_b, mask=None, chunk=16, clip=True):
    """Exact chunked 2D histogram — never materializes a full volume in RAM."""
    H = np.zeros((len(edges_a) - 1, len(edges_b) - 1), dtype=np.float64)
    nz = vol_a.shape[0]
    for z0 in range(0, nz, chunk):
        z1 = min(z0 + chunk, nz)
        a = np.asarray(vol_a[z0:z1], dtype=np.float64).ravel()
        b = np.asarray(vol_b[z0:z1], dtype=np.float64).ravel()

        keep = np.isfinite(a) & np.isfinite(b)
        if mask is not None:
            keep &= np.asarray(mask[z0:z1]).astype(bool).ravel()
        a, b = a[keep], b[keep]
        if a.size == 0:
            continue

        if clip:
            np.clip(a, edges_a[0], edges_a[-1], out=a)
            np.clip(b, edges_b[0], edges_b[-1], out=b)

        h, _, _ = np.histogram2d(a, b, bins=[edges_a, edges_b])
        H += h
    return H


H0 = joint_hist(series_a[0], series_b[0], edges_a, edges_b,
                mask=mask_of(0), chunk=CHUNK, clip=CLIP)

fig, ax = plt.subplots(figsize=(5.5, 4.5))
im = ax.imshow(np.log1p(H0), origin="lower", aspect="auto", extent=extent, cmap="viridis")
ax.set_xlabel(LABEL_B)
ax.set_ylabel(LABEL_A)
ax.set_title("t=0 joint histogram, log(1+counts)")
fig.colorbar(im, ax=ax)
plt.show()

print(f"counts: {H0.sum():.6g}   occupied bins: {(H0 > 0).sum()} / {H0.size}")
print(f"edge-bin fraction: {(H0[0].sum() + H0[-1].sum() + H0[:, 0].sum() + H0[:, -1].sum()) / H0.sum():.4%}")

## 5. All timepoints → `.npy`

In [ ]:
totals = []
for t in range(N_T):
    H = joint_hist(series_a[t], series_b[t], edges_a, edges_b,
                   mask=mask_of(t), chunk=CHUNK, clip=CLIP)
    np.save(os.path.join(OUT_DIR, f"hist_t{t:04d}.npy"), H)
    totals.append(float(H.sum()))
    print(f"t={t:04d}  counts={totals[-1]:.6g}")

meta = dict(source_a=os.path.abspath(VOL_A), source_b=os.path.abspath(VOL_B),
            axes_a=series_a.axes, axes_b=series_b.axes,
            label_a=LABEL_A, label_b=LABEL_B, n_timepoints=N_T, bins=BINS,
            range_a=[lo_a, hi_a], range_b=[lo_b, hi_b],
            mask=os.path.abspath(MASK) if MASK else None,
            clipped=bool(CLIP), counts_per_timepoint=totals,
            axis_order="rows = A, cols = B")
with open(os.path.join(OUT_DIR, "meta.json"), "w") as f:
    json.dump(meta, f, indent=2)

if CLIP and len(set(np.round(totals, 6))) > 1:
    print("\nNote: counts vary across timepoints — expected with a per-timepoint "
          "mask, otherwise worth checking.")
print(f"\nSaved {N_T} histograms to {OUT_DIR}/")

## 6. Normalize every timepoint against the first

The histograms are converted to probabilities first, then compared to `t=0`:

| mode | formula | use it when |
|---|---|---|
| `log2ratio` | `log2((P_t + eps) / (P_0 + eps))` | default — symmetric, a doubling reads as +1 whether the bin started dense or sparse |
| `diff` | `P_t - P_0` | you care about absolute mass moving; favors the high-density ridge |
| `ratio` | `P_t / (P_0 + eps)` | you want a plain multiplicative factor |
| `chi` | `(H_t - H_0) / sqrt(H_0)` | counts are genuinely photon/neutron-count-like and you want Poisson-scaled residuals |

In [ ]:
files = sorted(glob.glob(os.path.join(OUT_DIR, "hist_t*.npy")))
H = np.stack([np.load(f) for f in files]).astype(np.float64)

# Normalize BEFORE comparing.
P = H / H.sum(axis=(1, 2), keepdims=True)
ref = P[0]

if MODE == "diff":
    D = P - ref
elif MODE == "ratio":
    D = P / (ref + EPS)
elif MODE == "log2ratio":
    D = np.log2((P + EPS) / (ref + EPS))
elif MODE == "chi":
    D = (H - H[0]) / np.sqrt(np.maximum(H[0], 1.0))
else:
    raise ValueError(MODE)

# Bins where the reference is essentially empty are pure noise in the ratio
# modes — blank them so they don't hijack the color scale.
if MODE in ("ratio", "log2ratio") and REF_FLOOR > 0:
    D[:, ref < REF_FLOOR * ref.max()] = np.nan

cmp_dir = os.path.join(OUT_DIR, "compare_t0")
os.makedirs(cmp_dir, exist_ok=True)
np.save(os.path.join(cmp_dir, f"norm_{MODE}_stack.npy"), D)
np.save(os.path.join(cmp_dir, "prob_stack.npy"), P)
for t in range(N_T):
    np.save(os.path.join(cmp_dir, f"norm_{MODE}_t{t:04d}.npy"), D[t])

print(f"{MODE} stack: {D.shape} -> {cmp_dir}/")

## 7. Scalar metrics per timepoint

In [ ]:
def compare_metrics(Pt, Q):
    """Pt = timepoint t, Q = reference t0, both summing to 1."""
    tv = 0.5 * np.abs(Pt - Q).sum()
    M = 0.5 * (Pt + Q)
    with np.errstate(divide="ignore", invalid="ignore"):
        kl_pm = np.where(Pt > 0, Pt * np.log2(Pt / M), 0.0).sum()
        kl_qm = np.where(Q > 0, Q * np.log2(Q / M), 0.0).sum()
    jsd = 0.5 * (kl_pm + kl_qm)
    corr = np.corrcoef(Pt.ravel(), Q.ravel())[0, 1]
    return tv, jsd, corr


def mutual_information(Pt):
    """MI of the joint distribution — how tightly the two modalities couple."""
    pa = Pt.sum(axis=1, keepdims=True)
    pb = Pt.sum(axis=0, keepdims=True)
    outer = pa * pb
    nz = (Pt > 0) & (outer > 0)
    return float((Pt[nz] * np.log2(Pt[nz] / outer[nz])).sum())


rows = []
for t in range(N_T):
    tv, jsd, corr = compare_metrics(P[t], ref)
    rows.append((t, tv, jsd, corr, mutual_information(P[t])))

table = np.array(rows)
np.savetxt(os.path.join(cmp_dir, "metrics.csv"), table, delimiter=",",
           header="t,total_variation,jsd_bits,pearson_r,mi_bits", comments="")

try:
    import pandas as pd
    df = pd.DataFrame(table, columns=["t", "total_variation", "jsd_bits",
                                      "pearson_r", "mi_bits"])
    df["t"] = df["t"].astype(int)
    display(df.style.format({"total_variation": "{:.5f}", "jsd_bits": "{:.5f}",
                             "pearson_r": "{:.5f}", "mi_bits": "{:.5f}"}))
except ImportError:
    print(f"{'t':>5} {'TV':>10} {'JSD':>10} {'corr':>9} {'MI':>9}")
    for r in rows:
        print(f"{int(r[0]):5d} {r[1]:10.5f} {r[2]:10.5f} {r[3]:9.5f} {r[4]:9.5f}")

Mutual information is worth watching on its own: it measures how tightly the two
modalities are coupled at each timepoint. A steady drop usually means either real
decoupling in the sample or registration drifting between the two datasets.

## 8. Panels — every timepoint relative to t=0

In [ ]:
diverging = MODE in ("diff", "log2ratio", "chi")
finite = D[np.isfinite(D)]
if diverging:
    v = np.percentile(np.abs(finite), 99.5)
    kw = dict(cmap="RdBu_r", vmin=-v, vmax=v)
else:
    kw = dict(cmap="viridis", vmin=np.percentile(finite, 0.5),
              vmax=np.percentile(finite, 99.5))

ncol = min(6, N_T)
nrow = int(np.ceil(N_T / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(3.1 * ncol, 3.1 * nrow), squeeze=False)
for i in range(nrow * ncol):
    ax = axes[i // ncol][i % ncol]
    if i >= N_T:
        ax.axis("off")
        continue
    im = ax.imshow(D[i], origin="lower", aspect="auto", extent=extent, **kw)
    ax.set_title(f"t={i}", fontsize=9)
    if i % ncol == 0:
        ax.set_ylabel(LABEL_A)
    if i // ncol == nrow - 1:
        ax.set_xlabel(LABEL_B)
fig.colorbar(im, ax=axes, shrink=0.6, label=MODE)
fig.savefig(os.path.join(cmp_dir, f"panels_{MODE}.png"), dpi=140, bbox_inches="tight")
plt.show()

## 9. Temporal trends

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ts = table[:, 0]
ax.plot(ts, table[:, 1], "o-", label="total variation")
ax.plot(ts, table[:, 2], "s-", label="JSD (bits)")
ax.set_xlabel("timepoint")
ax.set_ylabel("divergence from t=0")

ax2 = ax.twinx()
ax2.plot(ts, table[:, 4], "^--", color="gray", label="MI (bits)")
ax2.set_ylabel("mutual information (bits)")

ax.legend(loc="upper left")
ax2.legend(loc="lower right")
fig.tight_layout()
fig.savefig(os.path.join(cmp_dir, "temporal_metrics.png"), dpi=140)
plt.show()

## 10. Inspect a single timepoint

Change `T_SHOW` and re-run to look at one comparison at full size, with the raw
histogram beside it for context.

In [ ]:
T_SHOW = min(N_T - 1, 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
a0 = axes[0].imshow(np.log1p(H[T_SHOW]), origin="lower", aspect="auto",
                    extent=extent, cmap="viridis")
axes[0].set_title(f"t={T_SHOW} joint histogram, log(1+counts)")
fig.colorbar(a0, ax=axes[0])

a1 = axes[1].imshow(D[T_SHOW], origin="lower", aspect="auto", extent=extent, **kw)
axes[1].set_title(f"t={T_SHOW} vs t=0 ({MODE})")
fig.colorbar(a1, ax=axes[1])

for ax in axes:
    ax.set_xlabel(LABEL_B)
    ax.set_ylabel(LABEL_A)
fig.tight_layout()
plt.show()

## 11. Trend inside a region of interest

The lithium signature is a rectangle in intensity space: low X-ray attenuation,
high neutron attenuation. Set the bounds in **intensity units**, not bin indices,
so they survive a change of `BINS`.

`None` means "open to the edge of the binned range". The bounds are snapped to bin
edges so that the histogram-based count and the voxel-based count in section 12
agree exactly.

Two things to watch in this corner of the plot:

* **Edge bins.** If clipping was on, every voxel above the range maximum was piled
  into the last row. An ROI touching that row absorbs all of it. The cell warns
  when this happens — it isn't automatically wrong (those voxels really are
  high-neutron), just be aware of what you are counting.
* **Low reference counts.** `log2ratio` is violently noisy where `t=0` had almost
  nothing, which is what the dark speckle in the top-left of your panels is. The
  ROI *fraction* below is computed from probabilities, not ratios, so it is immune
  to this — but don't read the speckle itself as signal.

In [ ]:
# --- ROI in intensity units -------------------------------------------------
ROI_A = (30000, None)      # neutron (rows): (lo, hi), None = open to range edge
ROI_B = (None, 15000)      # X-ray   (cols)

ca = 0.5 * (edges_a[:-1] + edges_a[1:])      # bin centres
cb = 0.5 * (edges_b[:-1] + edges_b[1:])

sel_a = np.ones(BINS, bool)
if ROI_A[0] is not None:
    sel_a &= ca >= ROI_A[0]
if ROI_A[1] is not None:
    sel_a &= ca <= ROI_A[1]
sel_b = np.ones(BINS, bool)
if ROI_B[0] is not None:
    sel_b &= cb >= ROI_B[0]
if ROI_B[1] is not None:
    sel_b &= cb <= ROI_B[1]

if not sel_a.any() or not sel_b.any():
    raise ValueError("ROI is empty - check the bounds against the printed ranges")

roi = np.outer(sel_a, sel_b)
ia, ib = np.flatnonzero(sel_a), np.flatnonzero(sel_b)

# Snap to bin edges; open ends become infinite so voxel and bin counts match
# the clipping convention used when the histograms were built.
a_lo = -np.inf if (ia[0] == 0 and CLIP) else edges_a[ia[0]]
a_hi = np.inf if (ia[-1] == BINS - 1 and CLIP) else edges_a[ia[-1] + 1]
b_lo = -np.inf if (ib[0] == 0 and CLIP) else edges_b[ib[0]]
b_hi = np.inf if (ib[-1] == BINS - 1 and CLIP) else edges_b[ib[-1] + 1]

print(f"ROI covers {roi.sum()} bins "
      f"({len(ia)} x {len(ib)} of {BINS} x {BINS})")
print(f"  {LABEL_A}: [{a_lo:.6g}, {a_hi:.6g})")
print(f"  {LABEL_B}: [{b_lo:.6g}, {b_hi:.6g})")
for touching, which in ((ia[0] == 0, f"{LABEL_A} min"), (ia[-1] == BINS - 1, f"{LABEL_A} max"),
                        (ib[0] == 0, f"{LABEL_B} min"), (ib[-1] == BINS - 1, f"{LABEL_B} max")):
    if touching and CLIP:
        print(f"  ! touches the {which} edge bin - includes all clipped out-of-range voxels")

# --- per-timepoint trend ----------------------------------------------------
frac = P[:, roi].sum(axis=1)                 # fraction of sampled voxels in ROI
cnt = H[:, roi].sum(axis=1)                  # raw voxel counts
w = P[:, roi]
with np.errstate(invalid="ignore"):
    cen_a = (w * ca[ia][:, None].repeat(len(ib), 1).ravel()).sum(axis=1) / frac
    cen_b = (w * cb[ib][None, :].repeat(len(ia), 0).ravel()).sum(axis=1) / frac

roi_trend = np.column_stack([np.arange(N_T), cnt, frac, frac / frac[0], cen_a, cen_b])
np.savetxt(os.path.join(cmp_dir, "roi_trend.csv"), roi_trend, delimiter=",",
           header="t,voxels,fraction,fraction_rel_t0,centroid_a,centroid_b", comments="")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].imshow(np.log1p(H[0]), origin="lower", aspect="auto", extent=extent, cmap="viridis")
axes[0].add_patch(plt.Rectangle(
    (max(b_lo, edges_b[0]), max(a_lo, edges_a[0])),
    min(b_hi, edges_b[-1]) - max(b_lo, edges_b[0]),
    min(a_hi, edges_a[-1]) - max(a_lo, edges_a[0]),
    fill=False, ec="red", lw=1.8))
axes[0].set(xlabel=LABEL_B, ylabel=LABEL_A, title="ROI on the t=0 histogram")

axes[1].plot(np.arange(N_T), 100 * frac, "o-")
axes[1].set(xlabel="timepoint", ylabel="% of sampled voxels in ROI",
            title="ROI population over time")
axes[1].grid(alpha=0.3)

axes[2].plot(np.arange(N_T), cen_b, "o-", label=f"{LABEL_B} centroid")
ax2 = axes[2].twinx()
ax2.plot(np.arange(N_T), cen_a, "s--", color="tab:red", label=f"{LABEL_A} centroid")
axes[2].set(xlabel="timepoint", ylabel=f"{LABEL_B} centroid")
ax2.set_ylabel(f"{LABEL_A} centroid")
axes[2].set_title("ROI centroid drift")
axes[2].legend(loc="upper left")
ax2.legend(loc="lower right")

fig.tight_layout()
fig.savefig(os.path.join(cmp_dir, "roi_trend.png"), dpi=140)
plt.show()

print(f"\nROI fraction: t=0 {100 * frac[0]:.4f}%  ->  "
      f"min {100 * frac.min():.4f}% (t={frac.argmin()})  "
      f"max {100 * frac.max():.4f}% (t={frac.argmax()})")

The population curve is the one to line up against your cycling protocol. The
centroid panel is a different question — it tells you whether the material inside
the ROI is *changing character* (the cluster sliding in intensity space) as opposed
to simply *growing and shrinking* at fixed composition. Those two look identical in
the panel plots and quite different here.

## 12. Segment the corresponding voxels

Same rectangle, applied back to the volumes: a voxel is labelled when its neutron
value and its X-ray value both fall inside the ROI. Written straight into a
memory-mapped output TIFF chunk by chunk, so nothing large is held in RAM.

The printed cross-check must read `True` — it confirms the voxels selected here are
exactly the voxels counted in the histogram bins above.

In [ ]:
SEG_PATH = os.path.join(OUT_DIR, "segmentation_roi.tif")
MIN_COMPONENT = 0          # drop connected blobs smaller than this many voxels
                           # (0 = off; needs scipy)
SEG_VALUE = 255            # 255 keeps ImageJ happy; use 1 for a plain 0/1 mask

Z, Y, X = series_a.vol_shape
seg = tifffile.memmap(SEG_PATH, shape=(N_T, Z, Y, X), dtype=np.uint8)

if MIN_COMPONENT > 0:
    from scipy import ndimage

seg_counts = []
for t in range(N_T):
    va, vb, m = series_a[t], series_b[t], mask_of(t)
    vol = np.zeros((Z, Y, X), bool)
    for z0 in range(0, Z, CHUNK):
        z1 = min(z0 + CHUNK, Z)
        a = np.asarray(va[z0:z1], dtype=np.float64)
        b = np.asarray(vb[z0:z1], dtype=np.float64)
        sel = (np.isfinite(a) & np.isfinite(b)
               & (a >= a_lo) & (a < a_hi) & (b >= b_lo) & (b < b_hi))
        if m is not None:
            sel &= np.asarray(m[z0:z1]).astype(bool)
        vol[z0:z1] = sel

    if MIN_COMPONENT > 0:
        lab, n = ndimage.label(vol)
        if n:
            sizes = np.bincount(lab.ravel())
            sizes[0] = 0
            vol = np.isin(lab, np.flatnonzero(sizes >= MIN_COMPONENT))

    seg[t] = vol.astype(np.uint8) * SEG_VALUE
    seg_counts.append(int(vol.sum()))
    print(f"t={t:04d}  voxels={seg_counts[-1]:>10d}  "
          f"({100 * seg_counts[-1] / vol.size:.4f}% of volume)")

seg.flush()
seg_counts = np.array(seg_counts, dtype=np.int64)

if MIN_COMPONENT == 0:
    agree = np.array_equal(seg_counts, cnt.astype(np.int64))
    print(f"\ncross-check vs histogram ROI counts: {agree}")
    if not agree:
        print("  mismatch:", seg_counts - cnt.astype(np.int64))
else:
    print(f"\nremoved {int(cnt.sum() - seg_counts.sum())} voxels in small components")

print(f"segmentation -> {SEG_PATH}  (T, Z, Y, X) uint8")

`MIN_COMPONENT` is off by default because it breaks the cross-check by design. Turn
it on once you trust the ROI and want isolated noise voxels gone — but note it
treats each timepoint independently, so a blob flickering above and below the
threshold will pop in and out of the count.

### Overlay on a slice

In [ ]:
T_SEG, Z_SEG = 0, Z // 2

sl_a = np.asarray(series_a[T_SEG][Z_SEG:Z_SEG + 1], dtype=np.float64)[0]
sl_b = np.asarray(series_b[T_SEG][Z_SEG:Z_SEG + 1], dtype=np.float64)[0]
sl_m = np.asarray(seg[T_SEG, Z_SEG]) > 0

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, sl, lab in ((axes[0], sl_a, LABEL_A), (axes[1], sl_b, LABEL_B)):
    ax.imshow(sl, cmap="gray")
    ax.set_title(f"{lab}  t={T_SEG}, z={Z_SEG}")
    if sl_m.any():
        ax.contour(sl_m, levels=[0.5], colors="red", linewidths=0.8)
    ax.axis("off")

axes[2].imshow(sl_m, cmap="magma")
axes[2].set_title(f"ROI segmentation ({sl_m.sum()} voxels in slice)")
axes[2].axis("off")

fig.tight_layout()
fig.savefig(os.path.join(cmp_dir, f"segmentation_overlay_t{T_SEG}_z{Z_SEG}.png"), dpi=140)
plt.show()

### Segmented volume over time

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(np.arange(N_T), seg_counts, "o-", color="tab:purple")
ax.set(xlabel="timepoint", ylabel="segmented voxels",
       title="ROI volume over time")
ax.grid(alpha=0.3)

if "VOXEL_SIZE_MM" in dir() and VOXEL_SIZE_MM:
    ax2 = ax.twinx()
    ax2.plot(np.arange(N_T), seg_counts * VOXEL_SIZE_MM ** 3, alpha=0)
    ax2.set_ylabel("volume (mm$^3$)")

fig.tight_layout()
fig.savefig(os.path.join(cmp_dir, "roi_volume.png"), dpi=140)
plt.show()

## 13. Binary mask volume

A clean 0/1 `uint8` volume, `(T, Z, Y, X)`, same grid as the input: 1 inside the
ROI, 0 everywhere else, every timepoint present. Section 12's file uses 255 for
ImageJ's benefit — this one is strictly binary for arithmetic (multiply it against
a volume, sum it, feed it to a labeller).

In [ ]:
BINARY_PATH = os.path.join(OUT_DIR, "roi_binary_mask.tif")

binmask = tifffile.memmap(BINARY_PATH, shape=(N_T, Z, Y, X), dtype=np.uint8)
for t in range(N_T):
    for z0 in range(0, Z, CHUNK):
        z1 = min(z0 + CHUNK, Z)
        binmask[t, z0:z1] = (np.asarray(seg[t, z0:z1]) > 0).astype(np.uint8)
binmask.flush()

vals = np.unique(np.asarray(binmask))
print(f"{BINARY_PATH}")
print(f"  shape {binmask.shape}  dtype {binmask.dtype}  values {vals}")
print(f"  ones per timepoint: {[int(binmask[t].sum()) for t in range(N_T)]}")
assert set(vals.tolist()) <= {0, 1}, "mask is not binary"
assert binmask.shape == (N_T, Z, Y, X) == (N_T,) + tuple(series_a.vol_shape)

# .npy alongside, if that suits downstream code better
np.save(os.path.join(OUT_DIR, "roi_binary_mask.npy"), np.asarray(binmask))

## 14. Where does the joint distribution actually change?

Rather than picking regions by eye, let the temporal variability pick them. Each
bin gets a variability score (peak-to-peak of its probability across time), bins
above a threshold are kept, and connected blobs among them become candidate
regions — the automatic generalisation of your hand-drawn ROI.

The count floor matters more than the threshold: a bin holding three voxels can
easily double, and without the floor the regions found are mostly noise.

In [ ]:
MIN_BIN_COUNT = 20         # a bin must reach this many voxels at some timepoint
VAR_PERCENTILE = 99.0      # keep bins above this percentile of variability
MIN_REGION_BINS = 10       # discard blobs smaller than this
MAX_REGIONS = 6

from scipy import ndimage

variability = np.ptp(P, axis=0)                    # peak-to-peak probability
eligible = H.max(axis=0) >= MIN_BIN_COUNT
score = np.where(eligible, variability, 0.0)

thr = np.percentile(score[eligible], VAR_PERCENTILE) if eligible.any() else np.inf
lab, n_lab = ndimage.label(score > thr)
print(f"{eligible.sum()} bins above the count floor, "
      f"{int((score > thr).sum())} above the variability threshold, {n_lab} blobs")

if not eligible.any():
    raise ValueError("no bin reaches MIN_BIN_COUNT - lower it, or use coarser BINS")

# Speckle in the histogram fragments blobs, and a percentile that is too strict
# leaves nothing to label. Close small gaps, then relax until something survives.
struct = np.ones((3, 3), bool)
attempts = [(VAR_PERCENTILE, MIN_REGION_BINS), (98.0, MIN_REGION_BINS),
            (95.0, MIN_REGION_BINS), (95.0, 3), (90.0, 3)]
keep, used = [], None
for p, minb in attempts:
    thr = np.percentile(score[eligible], p)
    lab, n_lab = ndimage.label(ndimage.binary_closing(score > thr, struct))
    sizes = np.bincount(lab.ravel())
    sizes[0] = 0
    keep = [i for i in np.argsort(sizes)[::-1] if sizes[i] >= minb][:MAX_REGIONS]
    if keep:
        used = (p, minb)
        break

if not keep:
    raise ValueError("no region found - lower VAR_PERCENTILE / MIN_REGION_BINS, "
                     "or check that anything actually changes over time")
if used != (VAR_PERCENTILE, MIN_REGION_BINS):
    print(f"  ! relaxed to percentile {used[0]}, min {used[1]} bins to find regions")

# renumber 1..K by total variability, largest first
regions = np.zeros_like(lab)
region_masks = []
for k, i in enumerate(sorted(keep, key=lambda i: -score[lab == i].sum()), start=1):
    regions[lab == i] = k
    region_masks.append(lab == i)
K = len(region_masks)
print(f"kept {K} regions of {[int(m.sum()) for m in region_masks]} bins")

# per-region trends and intensity-space description
reg_frac = np.stack([P[:, m].sum(axis=1) for m in region_masks])       # (K, T)
reg_cnt = np.stack([H[:, m].sum(axis=1) for m in region_masks])
rows_reg = []
for k, m in enumerate(region_masks):
    ii, jj = np.nonzero(m)
    rows_reg.append(dict(region=k + 1, bins=int(m.sum()),
                         a_range=(float(edges_a[ii.min()]), float(edges_a[ii.max() + 1])),
                         b_range=(float(edges_b[jj.min()]), float(edges_b[jj.max() + 1])),
                         frac_t0=float(reg_frac[k, 0]),
                         frac_min=float(reg_frac[k].min()),
                         frac_max=float(reg_frac[k].max())))
    r = rows_reg[-1]
    print(f"  region {k+1}: {LABEL_A} {r['a_range'][0]:.0f}-{r['a_range'][1]:.0f}, "
          f"{LABEL_B} {r['b_range'][0]:.0f}-{r['b_range'][1]:.0f}, "
          f"{100*r['frac_min']:.3f}% -> {100*r['frac_max']:.3f}%")

np.save(os.path.join(cmp_dir, "regions_labelmap.npy"), regions)
np.savetxt(os.path.join(cmp_dir, "region_fractions.csv"), reg_frac.T, delimiter=",",
           header=",".join(f"region_{k+1}" for k in range(K)), comments="")

In [ ]:
cmap_reg = plt.get_cmap("tab10")
reg_colors = [cmap_reg(k % 10) for k in range(K)]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

im = axes[0].imshow(np.log1p(np.where(eligible, variability, np.nan)), origin="lower",
                    aspect="auto", extent=extent, cmap="magma")
axes[0].set(xlabel=LABEL_B, ylabel=LABEL_A, title="temporal variability, log(1+ptp)")
fig.colorbar(im, ax=axes[0])

overlay = np.full(regions.shape + (4,), 0.0)
for k in range(K):
    overlay[region_masks[k]] = reg_colors[k]
axes[1].imshow(np.log1p(H[0]), origin="lower", aspect="auto", extent=extent, cmap="gray")
axes[1].imshow(overlay, origin="lower", aspect="auto", extent=extent)
for k in range(K):
    ii, jj = np.nonzero(region_masks[k])
    axes[1].annotate(str(k + 1), (edges_b[int(jj.mean())], edges_a[int(ii.mean())]),
                     color="white", fontsize=11, fontweight="bold")
axes[1].set(xlabel=LABEL_B, ylabel=LABEL_A, title="detected regions on t=0")

for k in range(K):
    axes[2].plot(np.arange(N_T), 100 * reg_frac[k], "o-", color=reg_colors[k],
                 label=f"region {k+1}")
axes[2].set(xlabel="timepoint", ylabel="% of sampled voxels", title="population per region")
axes[2].legend(fontsize=8)
axes[2].grid(alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(cmp_dir, "regions_overview.png"), dpi=140)
plt.show()

### Region kymograph

Regions on one axis, time on the other, colour = change relative to `t=0`. Compact
enough to sit next to a voltage curve, and it makes anti-correlated pairs obvious —
one region emptying as another fills is the signature of material moving between
two states rather than appearing or vanishing.

In [ ]:
with np.errstate(divide="ignore", invalid="ignore"):
    kymo = np.log2(reg_frac / reg_frac[:, [0]])
kymo[~np.isfinite(kymo)] = np.nan

v = np.nanpercentile(np.abs(kymo), 99) or 1.0
fig, ax = plt.subplots(figsize=(1.1 + 0.45 * N_T, 1.2 + 0.5 * K))
im = ax.imshow(kymo, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v,
               extent=[-0.5, N_T - 0.5, K + 0.5, 0.5])
ax.set_yticks(np.arange(1, K + 1))
ax.set(xlabel="timepoint", ylabel="region", title="log2 change vs t=0")
fig.colorbar(im, ax=ax, label="log2 ratio")
fig.tight_layout()
fig.savefig(os.path.join(cmp_dir, "region_kymograph.png"), dpi=140)
plt.show()

if K > 1:
    C = np.corrcoef(reg_frac)
    print("region cross-correlation (population over time):")
    for k in range(K):
        print("  " + "  ".join(f"{C[k, j]:+.2f}" for j in range(K)))

### Marginal kymographs

The same idea one dimension down: each modality's 1D histogram stacked against
time. Cheap, and it separates a shift in one modality from a shift in both — the
joint view can hide which axis is moving.

In [ ]:
marg_a = P.sum(axis=2)              # (T, BINS) neutron marginal
marg_b = P.sum(axis=1)              # (T, BINS) X-ray marginal

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, m, c, lab in ((axes[0], marg_a, ca, LABEL_A), (axes[1], marg_b, cb, LABEL_B)):
    with np.errstate(divide="ignore", invalid="ignore"):
        d = np.log2(m / m[0])
    d[~np.isfinite(d)] = np.nan
    vv = np.nanpercentile(np.abs(d), 99) or 1.0
    im = ax.imshow(d.T, origin="lower", aspect="auto", cmap="RdBu_r", vmin=-vv, vmax=vv,
                   extent=[-0.5, N_T - 0.5, c[0], c[-1]])
    ax.set(xlabel="timepoint", ylabel=lab, title=f"{lab} marginal, log2 vs t=0")
    fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(os.path.join(cmp_dir, "marginal_kymographs.png"), dpi=140)
plt.show()

### Fixed-grid kymograph

Instead of a handful of detected regions, tile the whole joint space into square
cells of a fixed width in gray levels (1000 by default) and give every occupied
cell its own row. Nothing is selected or thresholded away — you see the whole
distribution evolve at once, at the cost of a taller plot.

Two ways to build it:

* `"aggregate"` (default) sums the existing 256×256 bins into grid cells. Instant,
  but a fine bin straddling a grid boundary is assigned whole to one side, so cell
  edges are fuzzy by up to one fine bin width (printed below as a percentage of
  `GRID`). Below ~10% this is irrelevant for reading trends; above it the cell sums
  a wide intensity range and the cell warns you to switch.
* `"recompute"` re-reads the volumes with edges placed exactly on multiples of
  `GRID`. Exact, and costs one more pass over the data.

In [ ]:
GRID = 1000                # cell width in gray levels, both axes
GRID_SOURCE = "aggregate"  # "aggregate" (fast) | "recompute" (exact)
MIN_CELL_COUNT = 20        # a cell must reach this many voxels at some timepoint
GRID_SORT = "position"     # "position" (by neutron band) | "variability"

ia_g = np.floor(ca / GRID).astype(int)
ib_g = np.floor(cb / GRID).astype(int)
la, na = ia_g.min(), ia_g.max() - ia_g.min() + 1
lb, nb = ib_g.min(), ib_g.max() - ib_g.min() + 1
gedges_a = np.arange(la, la + na + 1) * float(GRID)
gedges_b = np.arange(lb, lb + nb + 1) * float(GRID)

if GRID_SOURCE == "aggregate":
    Ma = np.zeros((na, BINS)); Ma[ia_g - la, np.arange(BINS)] = 1
    Mb = np.zeros((nb, BINS)); Mb[ib_g - lb, np.arange(BINS)] = 1
    Hg = np.stack([Ma @ H[t] @ Mb.T for t in range(N_T)])
    w_a = (edges_a[-1] - edges_a[0]) / BINS
    w_b = (edges_b[-1] - edges_b[0]) / BINS
    fuzz = max(w_a, w_b) / GRID
    print(f"aggregated from the saved stack; cell edges fuzzy by up to "
          f"{100 * w_a / GRID:.1f}% ({LABEL_A}) and {100 * w_b / GRID:.1f}% ({LABEL_B}) of GRID")
    if fuzz > 0.10:
        print(f"  ! fine bins are {100 * fuzz:.0f}% of GRID - cell boundaries are coarse. "
              f"Use GRID_SOURCE = 'recompute', or raise BINS.")
elif GRID_SOURCE == "recompute":
    Hg = np.stack([joint_hist(series_a[t], series_b[t], gedges_a, gedges_b,
                              mask=mask_of(t), chunk=CHUNK, clip=CLIP)
                   for t in range(N_T)])
    print("recomputed from the volumes on exact grid edges")
else:
    raise ValueError(GRID_SOURCE)

print(f"grid: {na} x {nb} cells of {GRID} gray levels "
      f"({gedges_a[0]:.0f}-{gedges_a[-1]:.0f} {LABEL_A}, "
      f"{gedges_b[0]:.0f}-{gedges_b[-1]:.0f} {LABEL_B})")

Pg = Hg / Hg.sum(axis=(1, 2), keepdims=True)
occ = Hg.max(axis=0) >= MIN_CELL_COUNT
ci, cj = np.nonzero(occ)
print(f"{occ.sum()} of {occ.size} cells occupied (>= {MIN_CELL_COUNT} voxels)")
if occ.sum() == 0:
    raise ValueError("no cell reaches MIN_CELL_COUNT - lower it or widen GRID")

series = Pg[:, ci, cj].T                                   # (n_cells, T)
with np.errstate(divide="ignore", invalid="ignore"):
    kymo_g = np.log2(series / series[:, [0]])
kymo_g[~np.isfinite(kymo_g)] = np.nan

if GRID_SORT == "variability":
    order = np.argsort(-np.nanmax(np.abs(kymo_g), axis=1))
else:                                                       # by band, top-left first
    order = np.lexsort((cj, -ci))
ci, cj, kymo_g, series = ci[order], cj[order], kymo_g[order], series[order]
n_cells = len(ci)

np.save(os.path.join(cmp_dir, "grid_kymograph.npy"), kymo_g)
np.savetxt(os.path.join(cmp_dir, "grid_cells.csv"),
           np.column_stack([np.arange(n_cells), gedges_a[ci], gedges_a[ci + 1],
                            gedges_b[cj], gedges_b[cj + 1], Hg[:, ci, cj].T]),
           delimiter=",", comments="",
           header="row,a_lo,a_hi,b_lo,b_hi," + ",".join(f"counts_t{t}" for t in range(N_T)))

In [ ]:
fig = plt.figure(figsize=(15, max(5, 0.16 * n_cells)))
gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.5], wspace=0.25)

# left: where the cells are
ax0 = fig.add_subplot(gs[0])
ax0.imshow(np.log1p(H[0]), origin="lower", aspect="auto", extent=extent, cmap="gray")
for e in gedges_a:
    ax0.axhline(e, color="tab:cyan", lw=0.3, alpha=0.5)
for e in gedges_b:
    ax0.axvline(e, color="tab:cyan", lw=0.3, alpha=0.5)
span = np.nanmax(np.abs(kymo_g), axis=1)
sc = ax0.scatter(0.5 * (gedges_b[cj] + gedges_b[cj + 1]),
                 0.5 * (gedges_a[ci] + gedges_a[ci + 1]),
                 c=span, s=14, cmap="magma", edgecolors="none")
ax0.set(xlim=(edges_b[0], edges_b[-1]), ylim=(edges_a[0], edges_a[-1]),
        xlabel=LABEL_B, ylabel=LABEL_A, title=f"{GRID}-level grid, occupied cells")
fig.colorbar(sc, ax=ax0, label="max |log2 change|")

# right: the kymograph
ax1 = fig.add_subplot(gs[1])
v = np.nanpercentile(np.abs(kymo_g), 99) or 1.0
im = ax1.imshow(kymo_g, aspect="auto", cmap="RdBu_r", vmin=-v, vmax=v,
                interpolation="nearest", extent=[-0.5, N_T - 0.5, n_cells - 0.5, -0.5])
step = max(1, n_cells // 45)
rows = np.arange(0, n_cells, step)
ax1.set_yticks(rows)
ax1.set_yticklabels([f"{gedges_a[ci[r]]/1000:.0f}-{gedges_a[ci[r]+1]/1000:.0f}k | "
                     f"{gedges_b[cj[r]]/1000:.0f}k" for r in rows], fontsize=7)
if GRID_SORT == "position":
    for r in np.flatnonzero(np.diff(ci)) + 0.5:
        ax1.axhline(r, color="k", lw=0.4, alpha=0.4)
ax1.set(xlabel="timepoint", title=f"log2 change vs t=0, per {GRID}-level cell")
ax1.set_ylabel(f"{LABEL_A} band | {LABEL_B} band", labelpad=8)
fig.colorbar(im, ax=ax1, label="log2 ratio")

fig.savefig(os.path.join(cmp_dir, "grid_kymograph.png"), dpi=140, bbox_inches="tight")
plt.show()

worst = np.nanargmax(np.nanmax(np.abs(kymo_g), axis=1))
print(f"largest swing: {LABEL_A} {gedges_a[ci[worst]]:.0f}-{gedges_a[ci[worst]+1]:.0f}, "
      f"{LABEL_B} {gedges_b[cj[worst]]:.0f}-{gedges_b[cj[worst]+1]:.0f}  "
      f"({np.nanmax(np.abs(kymo_g[worst])):+.2f} log2)")

Row labels read *neutron band | X-ray band*, and with `GRID_SORT = "position"` the
rows are grouped into neutron bands with a rule between groups, so a horizontal
stripe of colour spanning one group means that whole neutron band is moving
regardless of X-ray value. `"variability"` instead puts the most active cells at
the top, which is the faster way to find what matters and the slower way to read
structure.

If the plot is too tall, raise `MIN_CELL_COUNT` — it drops sparsely populated
cells, which are also the noisiest — or widen `GRID` to 2000.

## 15. Decomposition of the histogram stack

Regions assume changes are spatially separate in intensity space. A decomposition
assumes instead that every histogram is a mixture of a few fixed patterns whose
weights move over time — closer to the truth when phases overlap.

**PCA** on the mean-centred stack gives orthogonal modes: component 1 is usually
"the main thing that changes", and its loading curve should track your cycling.
The maps are signed — red gains where blue loses.

**NMF** on the raw probabilities instead gives non-negative parts that sum to the
data, which read more like physical phases, at the cost of being non-unique.

In [ ]:
N_COMP = 3

Xf = P.reshape(N_T, -1)
mu = Xf.mean(axis=0)
U, S, Vt = np.linalg.svd(Xf - mu, full_matrices=False)
evr = S ** 2 / (S ** 2).sum()
comps = Vt[:N_COMP].reshape(N_COMP, BINS, BINS)
loads = U[:, :N_COMP] * S[:N_COMP]

print("explained variance:", "  ".join(f"PC{i+1} {100*evr[i]:.1f}%" for i in range(min(5, len(evr)))))

fig, axes = plt.subplots(2, N_COMP, figsize=(4.6 * N_COMP, 8))
for i in range(N_COMP):
    c = comps[i]
    vv = np.percentile(np.abs(c), 99.5) or 1.0
    im = axes[0][i].imshow(c, origin="lower", aspect="auto", extent=extent,
                           cmap="RdBu_r", vmin=-vv, vmax=vv)
    axes[0][i].set(title=f"PC{i+1} ({100*evr[i]:.1f}%)", xlabel=LABEL_B, ylabel=LABEL_A)
    fig.colorbar(im, ax=axes[0][i])
    axes[1][i].plot(np.arange(N_T), loads[:, i], "o-")
    axes[1][i].axhline(0, color="k", lw=0.6)
    axes[1][i].set(xlabel="timepoint", ylabel="loading", title=f"PC{i+1} over time")
    axes[1][i].grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(cmp_dir, "pca_components.png"), dpi=140)
plt.show()

np.save(os.path.join(cmp_dir, "pca_components.npy"), comps)
np.save(os.path.join(cmp_dir, "pca_loadings.npy"), loads)

In [ ]:
try:
    from sklearn.decomposition import NMF

    nmf = NMF(n_components=N_COMP, init="nndsvda", max_iter=800, random_state=0)
    W = nmf.fit_transform(Xf)                 # (T, N_COMP) weights
    Hc = nmf.components_.reshape(N_COMP, BINS, BINS)

    fig, axes = plt.subplots(2, N_COMP, figsize=(4.6 * N_COMP, 8))
    for i in range(N_COMP):
        im = axes[0][i].imshow(np.log1p(Hc[i] / Hc[i].max()), origin="lower",
                               aspect="auto", extent=extent, cmap="viridis")
        axes[0][i].set(title=f"NMF component {i+1}", xlabel=LABEL_B, ylabel=LABEL_A)
        fig.colorbar(im, ax=axes[0][i])
        axes[1][i].plot(np.arange(N_T), W[:, i], "o-", color="tab:green")
        axes[1][i].set(xlabel="timepoint", ylabel="weight", title=f"component {i+1} over time")
        axes[1][i].grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(cmp_dir, "nmf_components.png"), dpi=140)
    plt.show()

    np.save(os.path.join(cmp_dir, "nmf_components.npy"), Hc)
    np.save(os.path.join(cmp_dir, "nmf_weights.npy"), W)
except ImportError:
    print("scikit-learn not installed - skipping NMF (pip install scikit-learn)")

## 16. Put the regions back in the sample

Everything so far lives in intensity space, which says *what* changed but not
*where*. This maps each detected region back onto the voxels through a lookup
table on the bin indices, giving a `(T, Z, Y, X)` label volume: 0 for unassigned,
*k* for region *k*.

That is the answer to "which part of the cell is doing this" — open it in ImageJ
next to your reconstructions.

In [ ]:
LABEL_PATH = os.path.join(OUT_DIR, "region_labels.tif")

def bin_index(v, edges):
    idx = np.searchsorted(edges, v, side="right") - 1
    return np.clip(idx, 0, len(edges) - 2)

labels = tifffile.memmap(LABEL_PATH, shape=(N_T, Z, Y, X), dtype=np.uint8)
lut = regions.astype(np.uint8)               # (BINS, BINS) -> region id

label_counts = np.zeros((K, N_T), dtype=np.int64)
for t in range(N_T):
    va, vb, m = series_a[t], series_b[t], mask_of(t)
    for z0 in range(0, Z, CHUNK):
        z1 = min(z0 + CHUNK, Z)
        a = np.asarray(va[z0:z1], dtype=np.float64)
        b = np.asarray(vb[z0:z1], dtype=np.float64)
        ok = np.isfinite(a) & np.isfinite(b)
        if m is not None:
            ok &= np.asarray(m[z0:z1]).astype(bool)
        if not CLIP:
            ok &= ((a >= edges_a[0]) & (a <= edges_a[-1])
                   & (b >= edges_b[0]) & (b <= edges_b[-1]))
        out = np.zeros(a.shape, np.uint8)
        if ok.any():
            out[ok] = lut[bin_index(a[ok], edges_a), bin_index(b[ok], edges_b)]
        labels[t, z0:z1] = out
    for k in range(K):
        label_counts[k, t] = int((np.asarray(labels[t]) == k + 1).sum())
labels.flush()

agree = np.array_equal(label_counts, reg_cnt.astype(np.int64))
print(f"label volume -> {LABEL_PATH}  (T, Z, Y, X) uint8, {K} regions")
print(f"cross-check vs histogram region counts: {agree}")
if not agree:
    print(label_counts - reg_cnt.astype(np.int64))

In [ ]:
T_SHOW_LAB = [0, N_T // 2, N_T - 1]
Z_LAB = Z // 2

from matplotlib.colors import ListedColormap, BoundaryNorm
lab_cmap = ListedColormap([(0, 0, 0, 0)] + reg_colors)
norm = BoundaryNorm(np.arange(-0.5, K + 1), lab_cmap.N)

fig, axes = plt.subplots(1, len(T_SHOW_LAB), figsize=(5 * len(T_SHOW_LAB), 5))
for ax, t in zip(np.atleast_1d(axes), T_SHOW_LAB):
    ax.imshow(np.asarray(series_a[t][Z_LAB:Z_LAB + 1])[0], cmap="gray")
    ax.imshow(np.asarray(labels[t, Z_LAB]), cmap=lab_cmap, norm=norm, alpha=0.65)
    ax.set_title(f"t={t}, z={Z_LAB}")
    ax.axis("off")
handles = [plt.Line2D([], [], marker="s", ls="", color=reg_colors[k], label=f"region {k+1}")
           for k in range(K)]
fig.legend(handles=handles, loc="lower center", ncol=min(K, 6), frameon=False)
fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.savefig(os.path.join(cmp_dir, "region_labels_slices.png"), dpi=140)
plt.show()

### Where the regions sit, and whether they move

A region can hold a constant number of voxels while migrating through the sample.
The centre of mass and its spread separate "growing" from "moving" — worth checking
before reading a flat population curve as "nothing happened".

In [ ]:
com = np.full((K, N_T, 3), np.nan)
spread = np.full((K, N_T), np.nan)
for t in range(N_T):
    L = np.asarray(labels[t])
    for k in range(K):
        pts = np.argwhere(L == k + 1)
        if len(pts):
            com[k, t] = pts.mean(axis=0)
            spread[k, t] = np.sqrt(((pts - com[k, t]) ** 2).sum(axis=1).mean())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for k in range(K):
    for j, ax in enumerate(axes[:2]):
        ax.plot(np.arange(N_T), com[k, :, j], "o-", color=reg_colors[k], label=f"region {k+1}")
    axes[2].plot(np.arange(N_T), spread[k], "o-", color=reg_colors[k], label=f"region {k+1}")
axes[0].set(xlabel="timepoint", ylabel="centre of mass, Z", title="axial position")
axes[1].set(xlabel="timepoint", ylabel="centre of mass, Y", title="transverse position")
axes[2].set(xlabel="timepoint", ylabel="RMS radius (voxels)", title="spatial spread")
for ax in axes:
    ax.grid(alpha=0.3)
axes[2].legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(cmp_dir, "region_spatial_trends.png"), dpi=140)
plt.show()

## Outputs

```
hists/
  edges_a.npy, edges_b.npy      shared bin edges (BINS+1 values each)
  hist_t0000.npy ...            raw counts per timepoint, (BINS, BINS) float64
  meta.json                     ranges, mask, counts, provenance
  compare_t0/
    prob_stack.npy              (T, BINS, BINS) count-normalized
    norm_<mode>_stack.npy       (T, BINS, BINS) relative to t=0
    norm_<mode>_t0000.npy ...   same, one file per timepoint
    metrics.csv, roi_trend.csv, region_fractions.csv
    regions_labelmap.npy, pca_*.npy, nmf_*.npy
    grid_cells.csv, grid_kymograph.npy
    panels_<mode>.png, temporal_metrics.png
    roi_trend.png, roi_volume.png, segmentation_overlay_*.png
    regions_overview.png, region_kymograph.png, grid_kymograph.png
    marginal_kymographs.png
    pca_components.png, nmf_components.png
    region_labels_slices.png, region_spatial_trends.png
```

Re-running section 6 onward with a different `MODE` is cheap — it reads the saved
histograms and doesn't touch the volumes again.